# Deep Learning Weather Models

Train và so sánh LSTM, GRU cho bài toán dự báo `temperature_2m` từ chuỗi thời tiết hourly.

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import random

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import mean_absolute_error, median_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import GRU, LSTM, Dense, Dropout, Input
from tensorflow.keras.models import Sequential

sns.set_theme(style="whitegrid", palette="Set2")

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data/processed/Hanoi_final.csv").exists() and (PROJECT_ROOT.parent / "data/processed/Hanoi_final.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data/processed/Hanoi_final.csv"
OUTPUT_DIR = PROJECT_ROOT / "train_model/artifacts/dl"

TARGET_COLUMN = "temperature_2m"
HORIZON_HOURS = 1
SEQUENCE_LENGTH = 24
TEST_SIZE = 0.2
RANDOM_STATE = 42
BATCH_SIZE = 128
EPOCHS = 30

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print(DATA_PATH)

## 1. Đọc dữ liệu

In [ ]:
df = pd.read_csv(DATA_PATH)
df["time"] = pd.to_datetime(df["time"], errors="coerce")
df = df.dropna(subset=["time"]).sort_values("time").reset_index(drop=True)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")
print(f"Time: {df['time'].min()} -> {df['time'].max()}")
df.head()

## 2. Chọn feature và scale dữ liệu

In [ ]:
numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
feature_columns = [
    column
    for column in numeric_columns
    if column not in ["latitude", "longitude"]
]

if TARGET_COLUMN not in feature_columns:
    raise ValueError(f"Target column {TARGET_COLUMN} must be included in feature columns.")

split_index = int(len(df) * (1 - TEST_SIZE))

feature_scaler = MinMaxScaler()
target_scaler = MinMaxScaler()

feature_scaler.fit(df.loc[: split_index - 1, feature_columns])
target_scaler.fit(df.loc[: split_index - 1, [TARGET_COLUMN]])

features_scaled = feature_scaler.transform(df[feature_columns])
target_scaled = target_scaler.transform(df[[TARGET_COLUMN]]).reshape(-1)

print(f"Feature count: {len(feature_columns):,}")
print(f"Train boundary: {df.loc[split_index, 'time']}")

## 3. Tạo sequence cho LSTM/GRU

In [ ]:
def create_sequences(
    feature_values: np.ndarray,
    target_values: np.ndarray,
    timestamps: pd.Series,
    sequence_length: int,
    horizon: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    X, y, label_indices, label_times = [], [], [], []

    for end_index in range(sequence_length, len(feature_values) - horizon + 1):
        label_index = end_index + horizon - 1
        X.append(feature_values[end_index - sequence_length : end_index])
        y.append(target_values[label_index])
        label_indices.append(label_index)
        label_times.append(timestamps.iloc[label_index])

    return (
        np.asarray(X, dtype=np.float32),
        np.asarray(y, dtype=np.float32),
        np.asarray(label_indices),
        np.asarray(label_times),
    )


X, y, label_indices, label_times = create_sequences(
    feature_values=features_scaled,
    target_values=target_scaled,
    timestamps=df["time"],
    sequence_length=SEQUENCE_LENGTH,
    horizon=HORIZON_HOURS,
)

train_mask = label_indices < split_index
test_mask = label_indices >= split_index

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]
test_times = label_times[test_mask]

print(f"X_train: {X_train.shape}")
print(f"X_test : {X_test.shape}")

## 4. Hàm build model và đánh giá

In [ ]:
def build_lstm_model(input_shape: tuple[int, int]) -> tf.keras.Model:
    model = Sequential([
        Input(shape=input_shape),
        LSTM(64, return_sequences=True),
        Dropout(0.2),
        LSTM(32),
        Dropout(0.2),
        Dense(16, activation="relu"),
        Dense(1),
    ])
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return model


def build_gru_model(input_shape: tuple[int, int]) -> tf.keras.Model:
    model = Sequential([
        Input(shape=input_shape),
        GRU(64, return_sequences=True),
        Dropout(0.2),
        GRU(32),
        Dropout(0.2),
        Dense(16, activation="relu"),
        Dense(1),
    ])
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return model


def inverse_target(values: np.ndarray) -> np.ndarray:
    return target_scaler.inverse_transform(values.reshape(-1, 1)).reshape(-1)


def evaluate_predictions(y_true_scaled: np.ndarray, y_pred_scaled: np.ndarray) -> dict[str, float]:
    y_true = inverse_target(y_true_scaled)
    y_pred = inverse_target(y_pred_scaled)
    denominator = np.maximum(np.abs(y_true), 1e-8)
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "Median AE": median_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
        "MAPE (%)": np.mean(np.abs((y_true - y_pred) / denominator)) * 100,
    }


callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5),
]

## 5. Train LSTM

In [ ]:
lstm_model = build_lstm_model((SEQUENCE_LENGTH, len(feature_columns)))
lstm_history = lstm_model.fit(
    X_train,
    y_train,
    validation_split=0.1,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=False,
    callbacks=callbacks,
    verbose=1,
)

lstm_pred = lstm_model.predict(X_test).reshape(-1)
lstm_metrics = evaluate_predictions(y_test, lstm_pred)
lstm_metrics

## 6. Train GRU

In [ ]:
gru_model = build_gru_model((SEQUENCE_LENGTH, len(feature_columns)))
gru_history = gru_model.fit(
    X_train,
    y_train,
    validation_split=0.1,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=False,
    callbacks=callbacks,
    verbose=1,
)

gru_pred = gru_model.predict(X_test).reshape(-1)
gru_metrics = evaluate_predictions(y_test, gru_pred)
gru_metrics

## 7. So sánh LSTM và GRU

In [ ]:
comparison = pd.DataFrame([
    {"Model": "LSTM", **lstm_metrics},
    {"Model": "GRU", **gru_metrics},
]).set_index("Model").sort_values("RMSE")

comparison.round(4)

In [ ]:
best_model_name = comparison.index[0]
best_model = lstm_model if best_model_name == "LSTM" else gru_model
best_pred_scaled = lstm_pred if best_model_name == "LSTM" else gru_pred

predictions_df = pd.DataFrame({
    "time": test_times,
    "actual": inverse_target(y_test),
    "pred_lstm": inverse_target(lstm_pred),
    "pred_gru": inverse_target(gru_pred),
    "predicted_best": inverse_target(best_pred_scaled),
})
predictions_df["error"] = predictions_df["actual"] - predictions_df["predicted_best"]
predictions_df["absolute_error"] = predictions_df["error"].abs()

predictions_df.head()

## 8. Visualization và Error Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

pd.DataFrame(lstm_history.history)[["loss", "val_loss"]].plot(ax=axes[0])
axes[0].set_title("LSTM training history")

pd.DataFrame(gru_history.history)[["loss", "val_loss"]].plot(ax=axes[1])
axes[1].set_title("GRU training history")

plot_df = predictions_df.sample(min(5000, len(predictions_df)), random_state=RANDOM_STATE)
sns.scatterplot(data=plot_df, x="actual", y="predicted_best", alpha=0.35, ax=axes[2])
axes[2].plot([plot_df["actual"].min(), plot_df["actual"].max()], [plot_df["actual"].min(), plot_df["actual"].max()], color="red")
axes[2].set_title(f"Actual vs Predicted - {best_model_name}")

plt.tight_layout()

In [ ]:
predictions_df["hour"] = pd.to_datetime(predictions_df["time"]).dt.hour
predictions_df["month"] = pd.to_datetime(predictions_df["time"]).dt.month

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.lineplot(data=predictions_df.groupby("hour", as_index=False)["absolute_error"].mean(), x="hour", y="absolute_error", marker="o", ax=axes[0])
axes[0].set_title("DL MAE by hour")
sns.barplot(data=predictions_df.groupby("month", as_index=False)["absolute_error"].mean(), x="month", y="absolute_error", ax=axes[1])
axes[1].set_title("DL MAE by month")
plt.tight_layout()

## 9. Lưu model và kết quả

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

lstm_file = OUTPUT_DIR / f"lstm_{TARGET_COLUMN}_{timestamp}.keras"
gru_file = OUTPUT_DIR / f"gru_{TARGET_COLUMN}_{timestamp}.keras"
scaler_file = OUTPUT_DIR / f"dl_scalers_{TARGET_COLUMN}_{timestamp}.joblib"
metrics_file = OUTPUT_DIR / f"dl_metrics_{TARGET_COLUMN}_{timestamp}.json"
predictions_file = OUTPUT_DIR / f"dl_predictions_{TARGET_COLUMN}_{timestamp}.csv"

lstm_model.save(lstm_file)
gru_model.save(gru_file)
joblib.dump({
    "feature_scaler": feature_scaler,
    "target_scaler": target_scaler,
    "feature_columns": feature_columns,
    "target": TARGET_COLUMN,
    "sequence_length": SEQUENCE_LENGTH,
    "horizon_hours": HORIZON_HOURS,
    "best_model": best_model_name,
}, scaler_file)

metrics_payload = {
    "target": TARGET_COLUMN,
    "horizon_hours": HORIZON_HOURS,
    "sequence_length": SEQUENCE_LENGTH,
    "best_model": best_model_name,
    "metrics": comparison.to_dict(orient="index"),
}
metrics_file.write_text(json.dumps(metrics_payload, indent=2, ensure_ascii=False), encoding="utf-8")
predictions_df.to_csv(predictions_file, index=False)

print(lstm_file)
print(gru_file)
print(scaler_file)
print(metrics_file)
print(predictions_file)